# Stat reliability: results

Plots built from the files in `outputs/`. Run `python -m src.run_pipeline` first so those files are up to date.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path("../outputs") if Path("../outputs").exists() else Path("outputs")
curves  = pd.read_csv(OUT / "reliability_curves.csv")
summary = pd.read_csv(OUT / "reliability_summary.csv")
evals   = pd.read_csv(OUT / "shrinkage_eval.csv")
stats = list(summary["stat"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
summary

## 1. How fast does each stat stabilize? (split-half reliability)

Dots are measured reliability at each sample size; the line is the fitted curve r = N / (N + k).
Where a curve crosses the dashed line, half of the player-to-player spread is real skill.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for stat in stats:
    c = curves[curves["stat"] == stat].sort_values("games")
    k = float(summary.loc[summary["stat"] == stat, "k_games"].iloc[0])
    line, = ax.plot(np.linspace(0, c["games"].max() * 2, 200),
                    np.linspace(0, c["games"].max() * 2, 200) / (np.linspace(0, c["games"].max() * 2, 200) + k),
                    alpha=0.8, label=f"{stat} (k = {k:.0f} games)")
    ax.scatter(c["games"], c["reliability"], color=line.get_color(), zorder=3)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_xlabel("Games in each half-sample")
ax.set_ylabel("Reliability (split-half correlation)")
ax.set_ylim(0, 1)
ax.set_title("Reliability by sample size")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. Does shrinkage beat the alternatives? (out-of-sample error)

Each estimate uses a player's first N games and is scored on the rest of the season. Lower is better.
The most important comparison is **shrunk vs league average**.

In [ ]:
fig, axes = plt.subplots(1, len(stats), figsize=(4.2 * len(stats), 3.8))
axes = np.atleast_1d(axes)
for ax, stat in zip(axes, stats):
    e = evals[evals["stat"] == stat].sort_values("cutoff_games")
    ax.plot(e["cutoff_games"], e["mse_raw"], marker="o", label="Raw")
    ax.plot(e["cutoff_games"], e["mse_league_avg"], marker="o", label="League average")
    ax.plot(e["cutoff_games"], e["mse_shrunk"], marker="o", label="Shrunk")
    ax.set_title(stat); ax.set_xlabel("Games used to estimate skill")
axes[0].set_ylabel("Mean squared error on rest of season")
axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Same data as percentages: how much better is shrunk than each alternative?
e = evals.copy()
e["vs_league_avg_pct"] = 100 * (1 - e["mse_shrunk"] / e["mse_league_avg"])
e["vs_raw_pct"] = 100 * (1 - e["mse_shrunk"] / e["mse_raw"])
e.pivot(index="cutoff_games", columns="stat", values="vs_league_avg_pct").round(1)

## 3. Do the two reliability methods agree?

Attempts needed to reach 50% reliability, from the split-half study vs the beta-binomial model.

In [ ]:
x = np.arange(len(summary)); w = 0.38
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(x - w/2, summary["attempts_to_50_splithalf"], w, label="Split-half (Spearman-Brown)")
ax.bar(x + w/2, summary["attempts_to_50_betabinom"], w, label="Beta-binomial prior strength")
ax.set_xticks(x); ax.set_xticklabels(summary["stat"])
ax.set_ylabel("Attempts to 50% reliability")
ax.set_title("Two methods, two estimates")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()



## What I see

**Which stats stabilize fastest and slowest?**
Assist rate stabilizes fastest, FT% is next, and 3P% is by far the slowest. FT% needs only about 25 attempts to reach 50% reliability, while 3P% needs about 294, roughly 12x more. That fits intuition: a free throw is the same shot every time, whereas threes vary with defense, shot difficulty, and game context. Assist rate looks reliable mostly because roles differ so much between guards and big men, so I treat it as a measure of role as much as skill.

**Where does shrinkage help, and where does it barely beat the league average?**
Shrinkage helps a lot for FT%: 35% lower error than the league average after 10 games, rising to about 54% by game 40, because players genuinely differ. For 3P% it barely beats the league average early (about 1% at 10 games, about 9% at 40), which means early 3-point percentages tell us very little about the individual. The more important result is that raw 3P% after 10 games has about 4x the error of just guessing the league average, so reacting to a hot or cold start is worse than ignoring it.

**Why do the two reliability methods disagree, and what did I check?**
Split-half and beta-binomial estimates differ by up to about 2x (3P%: ~175 vs. ~294 attempts; FT%: ~61 vs. ~25). I tested shot volume of qualifying players and fitting the beta-binomial on the same players, and neither explained it. I then swept prior strengths on held-out games, and the beta-binomial values predicted best, so I use those as headline numbers. For 3P%, anything from about 175 to 500 predicts almost equally well, so the exact threshold is uncertain even though the ordering of stats is not.

**What would I do differently?**
I would build both methods in attempts from the start instead of converting games to attempts, fit priors by shot volume or position, and add confidence intervals so I could say which differences are real. I would also use play-by-play possessions and hold out a later season as a final test.